# 00 Inventory: what is already on the server

Read-only. Run all cells in the Real2Twin lab (JupyterLab, `nada-r2t-lab`). It checks the GPU and memory, the Python packages the Usable Space pipeline wants, and whether the team LLM serves a model that accepts images. It writes `/workspace/usable-space/out/inventory.json`: send me that file and I will plan the rest from facts, not guesses.

In [ ]:
import json, os, shutil, subprocess, importlib, importlib.metadata as md, platform, time
from pathlib import Path
OUT = Path('/workspace/usable-space/out'); OUT.mkdir(parents=True, exist_ok=True)
inv = {'when': time.strftime('%Y-%m-%d %H:%M:%S'), 'python': platform.python_version(), 'machine': platform.machine()}
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    return (r.stdout or r.stderr).strip()
inv['gpu'] = sh('nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader')
inv['memory'] = sh('free -h | head -2')
inv['disk_workspace'] = sh('df -h /workspace | tail -1')
print(json.dumps(inv, indent=2))

## Packages
Import name, then the version if it is installed.

In [ ]:
WANT = ['torch', 'transformers', 'cv2', 'skimage', 'trimesh', 'pandas', 'pyarrow', 'duckdb', 'ifcopenshell', 'dvc',
        'geopandas', 'shapely', 'pyproj', 'rasterio', 'fiona', 'momepy', 'osmnx', 'networkx', 'sklearn', 'xgboost',
        'rdflib', 'SPARQLWrapper', 'segment_anything', 'groundingdino', 'ultralytics', 'timm', 'requests', 'airflow']
pk = {}
for name in WANT:
    try:
        m = importlib.import_module(name)
        pk[name] = getattr(m, '__version__', 'installed')
    except Exception as e:
        pk[name] = f'missing ({type(e).__name__})'
for k, v in pk.items(): print(f'{k:18} {v}')
inv['packages'] = pk

## Models reachable through transformers
Checks that the classes exist; it downloads nothing.

In [ ]:
classes = {}
try:
    import transformers as T
    for c in ['SamModel', 'Sam2Model', 'GroundingDinoForObjectDetection', 'AutoModelForDepthEstimation', 'AutoModelForSemanticSegmentation', 'Qwen2_5_VLForConditionalGeneration', 'AutoModelForImageTextToText']:
        classes[c] = hasattr(T, c)
except Exception as e:
    classes['error'] = str(e)
print(classes)
inv['transformers_classes'] = classes
cache = Path(os.environ.get('HF_HOME', '/models/hf'))
inv['hf_cache'] = sorted(p.name for p in (cache / 'hub').glob('models--*')) if (cache / 'hub').exists() else []
print(inv['hf_cache'])

## Team LLM
Open WebUI is on port 3000 and vllm sits behind it. Put the vllm address in `VLLM_URL` if you know it (for example `http://172.18.0.2:8000`, the container IP from Portainer, or the host and its published port). Leave `None` to skip. The cell only lists the served models: a name with `VL` or `vision` in it takes images.

In [ ]:
VLLM_URL = None  # e.g. 'http://172.18.0.2:8000'
if VLLM_URL:
    import requests
    try:
        inv['vllm_models'] = [m['id'] for m in requests.get(VLLM_URL + '/v1/models', timeout=10).json()['data']]
    except Exception as e:
        inv['vllm_models'] = f'unreachable: {type(e).__name__}'
    print(inv['vllm_models'])
else:
    print('skipped')

In [ ]:
(OUT / 'inventory.json').write_text(json.dumps(inv, indent=2), encoding='utf-8')
print('written', OUT / 'inventory.json')